# Pratica - Lezione 1: Introduzione all'IA avanzata e la tua prima rete neurale

## Cosa farai in questa pratica

In questa pratica applichi i due meccanismi che sostengono tutto il deep learning (*deep learning*): il calcolo del gradiente tramite la regola della catena (*backpropagation*) e l'addestramento di una rete con PyTorch. Svolgi due esercizi:

1. Completa a mano il calcolo del gradiente di un neurone con un solo peso, senza nessuna libreria di differenziazione automatica.
2. Costruisci e addestri un perceptron multistrato (MLP) in PyTorch per classificare un insieme di dati non lineare, e visualizzi il confine di decisione che apprende.

## Prerequisito

Prima di iniziare, ripassa la lezione 1 in diretta e i capitoli 1 e 2 del manuale ("Introduzione al deep learning avanzato" e "Fondamenti delle reti neurali"). Devi capire cos'è un neurone artificiale, cosa fa una funzione di attivazione e che ruolo ha la discesa del gradiente.

## Come lavorare su questo notebook

Ogni esercizio segue lo stesso schema: leggi il testo, completa le parti contrassegnate con `# TODO` nel codice ed esegui la cella. Apri la soluzione (collassata) solo se ti blocchi o se vuoi verificare la tua risposta; prova a risolvere ogni esercizio da solo prima di guardarla.

Apri questo notebook in Google Colab e inizia dall'esercizio 1.

## Esercizio 1 - Backpropagation a mano: il gradiente di un singolo neurone

**Contesto.** In DIH DATAlife lavoriamo con sensori dell'internet delle cose (IoT) installati in aziende agricole e forestali, che misurano variabili come l'umidità del suolo. Immagina un sensore che fornisce l'umidità normalizzata del suolo (`x`, tra 0 e 1) a un neurone che decide con quale probabilità attivare l'irrigazione. Questo neurone ha un unico peso `w` (senza bias, per mantenere semplice l'esempio), usa la funzione di attivazione sigmoide e si addestra confrontando la sua previsione con un obiettivo tramite la perdita quadratica (errore quadratico medio, MSE).

$$\hat{y} = \sigma(w \cdot x), \qquad \mathcal{L} = (\hat{y} - y_{objetivo})^2$$

Nella cella di codice seguente hai già scritti i dati, il peso iniziale, le funzioni ausiliarie (`sigmoide` e `derivada_sigmoide`) e il passo in avanti (*forward pass*) dentro il ciclo di addestramento. Ti manca da completare il passo all'indietro (*backward pass*): il calcolo del gradiente della perdita rispetto al peso `w`, applicando la regola della catena termine a termine, e l'aggiornamento del peso con la discesa del gradiente.

**Suggerimenti:**

- La regola della catena scompone il gradiente in tre derivate che si moltiplicano: `dL/dw = dL/dy_pred * dy_pred/dz * dz/dw`
- `dL/dy_pred` è la derivata della perdita quadratica rispetto alla previsione: `2 * (y_pred - y_objetivo)`
- `dy_pred/dz` è la derivata della sigmoide in `z`; hai già la funzione `derivada_sigmoide` scritta per calcolarla
- `dz/dw` è la derivata di `z = w * x` rispetto a `w`, che è semplicemente `x`
- L'aggiornamento della discesa del gradiente è: `w = w - eta * gradiente_w`

In [ ]:
import numpy as np

# --- Datos del sensor ---
# x: humedad normalizada del suelo (0 = seco, 1 = saturado)
# y_objetivo: probabilidad ideal de activar el riego para esa humedad
x = 2.0
y_objetivo = 1.0

# Peso inicial de la neurona (en un caso real empezaria en un valor aleatorio;
# aqui lo fijamos para que puedas verificar tus calculos a mano)
w = 0.3

# Tasa de aprendizaje (learning rate)
eta = 0.1

def sigmoide(z):
    """Funcion de activacion sigmoide: comprime cualquier numero real en (0, 1)."""
    return 1.0 / (1.0 + np.exp(-z))

def derivada_sigmoide(z):
    """Derivada de la sigmoide: f'(z) = f(z) * (1 - f(z))."""
    s = sigmoide(z)
    return s * (1.0 - s)

historial_perdida = []

for paso in range(5):
    # --- Pase hacia adelante (forward pass), ya escrito ---
    z = w * x
    y_pred = sigmoide(z)
    perdida = (y_pred - y_objetivo) ** 2
    historial_perdida.append(perdida)

    # --- Pase hacia atras (backward pass): completa los TODO ---

    # TODO 1: calcula dL/dy_pred, la derivada de la perdida cuadratica respecto a y_pred
    dL_dypred = ...

    # TODO 2: calcula dy_pred/dz, la derivada de la sigmoide en z (usa derivada_sigmoide)
    dypred_dz = ...

    # TODO 3: calcula dz/dw, la derivada de z = w * x respecto a w
    dz_dw = ...

    # TODO 4: combina las tres derivadas anteriores con la regla de la cadena
    # para obtener el gradiente de la perdida respecto al peso w
    gradiente_w = ...

    # TODO 5: actualiza w con descenso de gradiente (w = w - eta * gradiente)
    w = ...

    print(f"paso {paso}: w={w:.5f}  y_pred={y_pred:.5f}  perdida={perdida:.5f}  gradiente={gradiente_w:.5f}")

In [ ]:
import matplotlib.pyplot as plt

plt.plot(historial_perdida, marker="o")
plt.xlabel("Paso de entrenamiento")
plt.ylabel("Perdida")
plt.title("Evolucion de la perdida durante el entrenamiento a mano")
plt.show()

<details>
<summary>💡 Vedi soluzione</summary>

```python
import numpy as np

x = 2.0
y_objetivo = 1.0
w = 0.3
eta = 0.1

def sigmoide(z):
    return 1.0 / (1.0 + np.exp(-z))

def derivada_sigmoide(z):
    s = sigmoide(z)
    return s * (1.0 - s)

historial_perdida = []

for paso in range(5):
    # --- Passo in avanti ---
    z = w * x
    y_pred = sigmoide(z)
    perdida = (y_pred - y_objetivo) ** 2
    historial_perdida.append(perdida)

    # --- Passo all'indietro: regola della catena, termine a termine ---
    dL_dypred = 2 * (y_pred - y_objetivo)   # derivata della perdita rispetto a y_pred
    dypred_dz = derivada_sigmoide(z)        # derivata della sigmoide rispetto a z
    dz_dw = x                               # derivata di z = w * x rispetto a w

    # dL/dw = dL/dy_pred * dy_pred/dz * dz/dw
    gradiente_w = dL_dypred * dypred_dz * dz_dw

    # --- Discesa del gradiente ---
    w = w - eta * gradiente_w

    print(f"paso {paso}: w={w:.5f}  y_pred={y_pred:.5f}  perdida={perdida:.5f}  gradiente={gradiente_w:.5f}")
```

Se esegui questa soluzione, vedrai che `y_pred` si avvicina poco a poco a `1.0` e che la perdita diminuisce a ogni passo: questo conferma che il gradiente che hai calcolato punta nella direzione giusta per minimizzare l'errore. Quando più avanti addestrerai reti con migliaia di pesi, PyTorch calcolerà automaticamente questa stessa catena di derivate con `.backward()`.
</details>

## Esercizio 2 - Il tuo primo perceptron multistrato (MLP) con PyTorch

**Contesto.** Vuoi costruire un classificatore binario capace di separare due classi che non si possono tracciare con una sola linea retta. Userai il dataset `make_moons` di scikit-learn, che genera due "mezzelune" intrecciate: un esempio non lineare, affine per spirito al problema XOR (o esclusivo) che hai visto nella lezione 1.

Costruirai un perceptron multistrato (MLP), l'architettura di rete neurale più semplice con strati nascosti, e lo addestrerai con PyTorch. Hai già scritti il caricamento e la preparazione dei dati e il ciclo di addestramento completo (passo in avanti, calcolo della perdita, passo all'indietro e aggiornamento dei pesi). Ti manca da definire l'architettura del modello: gli strati che lo compongono e come sono collegati nel metodo `forward`.

**Suggerimenti:**

- Uno strato lineare in PyTorch si crea con `nn.Linear(n_entradas, n_salidas)`
- Tra uno strato e l'altro serve una funzione di attivazione non lineare; usa `nn.ReLU()`, come hai visto nel capitolo 2
- Essendo un problema di classificazione binaria (due classi), lo strato finale produce una sola uscita, che trasformi in probabilità con `nn.Sigmoid()`
- Nel metodo `forward`, concatena gli strati nello stesso ordine in cui li hai definiti in `__init__`: ingresso -> strato nascosto -> attivazione -> strato di uscita -> sigmoide

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split

torch.manual_seed(42)
np.random.seed(42)

# --- Generamos el dataset de las dos medias lunas ---
X, y = make_moons(n_samples=500, noise=0.2, random_state=42)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Convertimos los datos a tensores de PyTorch
X_train = torch.tensor(X_train, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32).view(-1, 1)
X_test = torch.tensor(X_test, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.float32).view(-1, 1)

# Visualizamos los datos de entrenamiento
plt.figure(figsize=(5, 4))
plt.scatter(X_train[:, 0], X_train[:, 1], c=y_train.squeeze(), cmap="coolwarm", edgecolors="k")
plt.title("Dataset make_moons")
plt.xlabel("x1")
plt.ylabel("x2")
plt.show()

In [ ]:
class MLP(nn.Module):
    """Perceptron multicapa con una capa oculta para clasificacion binaria."""

    def __init__(self, n_entradas=2, n_ocultas=16):
        super().__init__()
        # TODO: define las cuatro piezas del modelo
        # - self.capa1: capa lineal de n_entradas a n_ocultas
        # - self.activacion: una activacion ReLU
        # - self.capa2: capa lineal de n_ocultas a 1 (una sola salida, clasificacion binaria)
        # - self.sigmoide: activacion sigmoide para convertir la salida en probabilidad
        self.capa1 = ...
        self.activacion = ...
        self.capa2 = ...
        self.sigmoide = ...

    def forward(self, x):
        # TODO: encadena las capas definidas arriba en el orden correcto
        # entrada -> capa1 -> activacion -> capa2 -> sigmoide
        salida = ...
        return salida

modelo = MLP()
print(modelo)

In [ ]:
criterio = nn.BCELoss()  # entropia cruzada binaria, para clasificacion con dos clases
optimizador = torch.optim.Adam(modelo.parameters(), lr=0.01)

n_epocas = 200
historial_perdida = []

for epoca in range(n_epocas):
    # --- Pase hacia adelante ---
    y_pred = modelo(X_train)
    perdida = criterio(y_pred, y_train)
    historial_perdida.append(perdida.item())

    # --- Pase hacia atras y actualizacion ---
    optimizador.zero_grad()
    perdida.backward()
    optimizador.step()

    if (epoca + 1) % 20 == 0:
        print(f"epoca {epoca + 1}/{n_epocas}  perdida={perdida.item():.4f}")

plt.plot(historial_perdida)
plt.xlabel("Epoca")
plt.ylabel("Perdida (BCE)")
plt.title("Evolucion de la perdida durante el entrenamiento")
plt.show()

In [ ]:
with torch.no_grad():
    y_pred_test = modelo(X_test)
    y_pred_clase = (y_pred_test > 0.5).float()
    precision = (y_pred_clase == y_test).float().mean()

print(f"Precision (accuracy) en el conjunto de prueba: {precision.item():.2%}")

In [ ]:
def graficar_frontera_decision(modelo, X, y):
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.02), np.arange(y_min, y_max, 0.02))

    malla = torch.tensor(np.c_[xx.ravel(), yy.ravel()], dtype=torch.float32)
    with torch.no_grad():
        Z = modelo(malla)
    Z = Z.reshape(xx.shape)

    plt.figure(figsize=(6, 5))
    plt.contourf(xx, yy, Z, levels=20, cmap="coolwarm", alpha=0.7)
    plt.scatter(X[:, 0], X[:, 1], c=y.squeeze(), cmap="coolwarm", edgecolors="k")
    plt.title("Frontera de decision aprendida por el MLP")
    plt.xlabel("x1")
    plt.ylabel("x2")
    plt.show()

graficar_frontera_decision(modelo, X_test, y_test)

<details>
<summary>💡 Vedi soluzione</summary>

```python
class MLP(nn.Module):
    """Perceptron multistrato con uno strato nascosto per la classificazione binaria."""

    def __init__(self, n_entradas=2, n_ocultas=16):
        super().__init__()
        self.capa1 = nn.Linear(n_entradas, n_ocultas)  # ingresso -> strato nascosto
        self.activacion = nn.ReLU()                    # non linearità tra gli strati
        self.capa2 = nn.Linear(n_ocultas, 1)            # strato nascosto -> uscita (1 neurone)
        self.sigmoide = nn.Sigmoid()                    # converte l'uscita in probabilità

    def forward(self, x):
        x = self.capa1(x)
        x = self.activacion(x)
        x = self.capa2(x)
        salida = self.sigmoide(x)
        return salida

modelo = MLP()
print(modelo)
```

Osserva lo schema: lo strato nascosto (`capa1` + `activacion`) deforma lo spazio di ingresso fino a renderlo separabile, come hai visto nel capitolo 2 con l'esempio dello XOR; lo strato di uscita (`capa2` + `sigmoide`) deve solo tracciare un confine semplice su quello spazio già trasformato. Se esegui il ciclo di addestramento e il grafico della frontiera di decisione con questo modello, dovresti vedere una curva che separa le due mezzelune con una precisione vicina o superiore al 95 % sul set di test.
</details>

## Hai completato la pratica della lezione 1

Hai toccato con mano le due facce di una stessa medaglia. Nell'esercizio 1 hai calcolato un gradiente termine a termine, senza magia, solo la regola della catena applicata a un neurone. Nell'esercizio 2 hai visto come PyTorch automatizza esattamente lo stesso calcolo, per quanto complesso sia il modello, grazie ad `autograd`, e come uno strato nascosto con attivazione non lineare risolva un problema che nessuna retta può risolvere.

Questo schema in tre passi, passo in avanti, `backward()`, `optimizador.step()`, è quello che ripeterai in ogni architettura che costruirai durante il resto del corso: reti convoluzionali, reti ricorrenti, meccanismi di attenzione e modelli generativi partono tutti da questa stessa base.

**Salva una copia di questo notebook nel tuo Google Drive e ripassalo prima della prossima lezione.**